# Data Collection

This notebook handles all data collection for the F1 Race Result Predictor. It fetches race session data via the FastF1 API, retrieves historical and forecast weather from Open-Meteo, and writes per-race CSVs to the appropriate season directories.

---

## Contents

1. [Imports](#1-imports)
2. [Track Locations](#2-track-locations)
3. [ID Mappings](#3-id-mappings)
4. [Feature Engineering Helpers](#4-feature-engineering-helpers)
5. [Weather Fetching](#5-weather-fetching)
6. [Main Collection Loop (2022–2025)](#6-main-collection-loop-20222025)
7. [Single-Race Backfill](#7-single-race-backfill)
8. [Pre-Race Prediction Input](#8-pre-race-prediction-input)

---
## 1. Imports

In [ ]:
import csv
import requests
from datetime import date, timedelta

import fastf1
import pandas as pd

---
## 2. Track Locations

Latitude, longitude, and elevation for every circuit on the calendar (2022–2025). These coordinates are passed directly to the Open-Meteo API when fetching weather data.

In [ ]:
track_locations = {
    "sakhir":            {"latitude": 26.0325,  "longitude":  50.5106,  "elevation_m":    7},
    "jeddah":            {"latitude": 21.6319,  "longitude":  39.1044,  "elevation_m":    3},
    "melbourne":         {"latitude": -37.8497, "longitude": 144.9683,  "elevation_m":    9},
    "imola":             {"latitude": 44.3439,  "longitude":  11.7167,  "elevation_m":   47},
    "miami":             {"latitude": 25.9588,  "longitude": -80.2389,  "elevation_m":    2},
    "barcelona":         {"latitude": 41.57,    "longitude":   2.2611,  "elevation_m":   90},
    "monaco":            {"latitude": 43.7347,  "longitude":   7.4206,  "elevation_m":   52},
    "baku":              {"latitude": 40.3725,  "longitude":  49.8533,  "elevation_m":  -28},
    "montreal":          {"latitude": 45.5,     "longitude": -73.5225,  "elevation_m":   10},
    "silverstone":       {"latitude": 52.07,    "longitude":  -1.016,   "elevation_m":  140},
    "spielberg":         {"latitude": 47.2197,  "longitude":  15.7644,  "elevation_m":  660},
    "le castellet":      {"latitude": 43.2506,  "longitude":   5.7903,  "elevation_m":  400},
    "budapest":          {"latitude": 47.5833,  "longitude":  19.2486,  "elevation_m":  171},
    "spa-francorchamps": {"latitude": 50.4372,  "longitude":   5.9714,  "elevation_m":  443},
    "zandvoort":         {"latitude": 52.3882,  "longitude":   4.5407,  "elevation_m":    9},
    "monza":             {"latitude": 45.6156,  "longitude":   9.2811,  "elevation_m":  162},
    "marina bay":        {"latitude":  1.2914,  "longitude": 103.8641,  "elevation_m":   15},
    "suzuka":            {"latitude": 34.8431,  "longitude": 136.5411,  "elevation_m":   50},
    "austin":            {"latitude": 30.1328,  "longitude": -97.6411,  "elevation_m":  180},
    "mexico city":       {"latitude": 19.4042,  "longitude": -99.0901,  "elevation_m": 2250},
    "sao paulo":         {"latitude": -23.7036, "longitude": -46.6997,  "elevation_m":  750},
    "yas island":        {"latitude": 24.4672,  "longitude":  54.6031,  "elevation_m":   10},
    "las vegas":         {"latitude": 36.1416,  "longitude": -115.1719, "elevation_m":  610},
    "lusail":            {"latitude": 25.49,    "longitude":  51.4542,  "elevation_m":    5},
    "shanghai":          {"latitude": 31.3389,  "longitude": 121.2197,  "elevation_m":    5},
}

---
## 3. ID Mappings

Integer IDs for teams and drivers keep the CSVs compact and consistent across seasons. Team IDs are stable; drivers who switched teams retain the same ID. Note that AlphaTauri / RB / Racing Bulls and Alfa Romeo / Kick Sauber share IDs to reflect constructor continuity.

In [ ]:
team_ids = {
    "Red Bull Racing": 1,
    "Mercedes":        2,
    "Ferrari":         3,
    "McLaren":         4,
    "Alpine":          5,
    "AlphaTauri":      6,  # Renamed to RB / Racing Bulls from 2024
    "RB":              6,
    "Racing Bulls":    6,
    "Alfa Romeo":      7,  # Renamed to Kick Sauber from 2024
    "Kick Sauber":     7,
    "Haas F1 Team":    8,
    "Williams":        9,
    "Aston Martin":    10,
}

In [ ]:
# Frozen driver ID map covering all drivers across the 2022–2025 seasons.
# New entries should be appended at the end to avoid shifting existing IDs.
driver_ids = {
    "Charles Leclerc":       1,
    "Carlos Sainz":          2,
    "Lewis Hamilton":        3,
    "George Russell":        4,
    "Kevin Magnussen":       5,
    "Valtteri Bottas":       6,
    "Esteban Ocon":          7,
    "Yuki Tsunoda":          8,
    "Fernando Alonso":       9,
    "Guanyu Zhou":           10,
    "Mick Schumacher":       11,
    "Lance Stroll":          12,
    "Alexander Albon":       13,
    "Daniel Ricciardo":      14,
    "Lando Norris":          15,
    "Nicholas Latifi":       16,
    "Nico Hulkenberg":       17,
    "Sergio Perez":          18,
    "Max Verstappen":        19,
    "Pierre Gasly":          20,
    "Sebastian Vettel":      21,
    "Nyck De Vries":         22,
    "Logan Sargeant":        23,
    "Oscar Piastri":         24,
    "Liam Lawson":           25,
    "Oliver Bearman":        26,
    "Franco Colapinto":      27,
    "Jack Doohan":           28,
    "Gabriel Bortoleto":     30,
    "Isack Hadjar":          31,
    "Kimi Antonelli":        32,
}

---
## 4. Feature Engineering Helpers

Each function extracts one feature from the FastF1 session objects. All weather helpers average over the 4-hour window covering the race start, matching the index offset introduced by fetching ±1 day of hourly data (index `time.hour + 24` is the race start hour in the padded array).

### CSV Schema

| Column | Description |
|---|---|
| `DriverId` | Integer driver ID (see mapping above) |
| `TeamId` | Integer team ID (see mapping above) |
| `Season` | Championship year |
| `RoundNumber` | Round number within the season |
| `RacesInGEEra` | Cumulative race count since the 2022 ground-effect regulation change |
| `LocationId` | Integer circuit ID (assigned incrementally on first encounter) |
| `RainBefore` | Total rainfall (mm) in the 5 hours before race start |
| `RainDuring` | Total rainfall (mm) during the 4-hour race window |
| `WindSpeed` | Average wind speed (km/h) during the race window |
| `ApparentTemp` | Average apparent temperature (°C) during the race window |
| `RelativeHumidity` | Average relative humidity (%) during the race window |
| `StartingPos` | Grid position at race start |
| `QualifyingPos` | Official qualifying classification position |
| `GapToPole` | Gap to pole position (seconds), measured in the highest qualifying session reached |
| `TeammateQualifyingPos` | Teammate's qualifying position (0 if no teammate data available) |
| `Retired` | `True` if the driver did not finish (not `"Finished"` or `"Lapped"`) |
| `FinishingPos` | Race finishing position (target variable) |

In [ ]:
def calculate_wind_speed(weather, time):
    """Average wind speed (km/h) over the 4-hour race window."""
    data = weather["wind_speed_10m"]
    start = time.hour + 24
    return sum(data[start:start + 4]) / 4


def calculate_apparent_temp(weather, time):
    """Average apparent temperature (°C) over the 4-hour race window."""
    data = weather["apparent_temperature"]
    start = time.hour + 24
    return sum(data[start:start + 4]) / 4


def calculate_relative_humidity(weather, time):
    """Average relative humidity (%) over the 4-hour race window."""
    data = weather["relative_humidity_2m"]
    start = time.hour + 24
    return sum(data[start:start + 4]) / 4


def calculate_rain_amount_before(weather, time):
    """Total rainfall (mm) in the 5 hours immediately before race start."""
    data = weather["rain"]
    end = time.hour + 24
    return sum(data[end - 5:end])


def calculate_rain_amount_during(weather, time):
    """Total rainfall (mm) during the 4-hour race window."""
    data = weather["rain"]
    start = time.hour + 24
    return sum(data[start:start + 4])

In [ ]:
def calculate_gap_to_pole_position(qualifying):
    """
    Return a dict mapping driver number → gap to pole (as a Timedelta).

    Each driver is compared against the pole time of the highest qualifying
    session they participated in (Q3 > Q2 > Q1). If a driver's best time
    is faster than the Q3 pole (e.g. data anomaly), they are compared
    against their own session's pole instead.
    """
    q1, q2, q3 = qualifying.laps.split_qualifying_sessions()
    q1_pole = q1.pick_fastest()["LapTime"]
    q2_pole = q2.pick_fastest()["LapTime"]
    q3_pole = q3.pick_fastest()["LapTime"]

    pole_gaps = {}

    for _, driver in qualifying.results.iterrows():
        if pd.notna(driver["Q3"]):
            pole_gaps[driver["DriverNumber"]] = driver["Q3"] - q3_pole
        elif pd.notna(driver["Q2"]):
            gap = driver["Q2"] - q3_pole
            pole_gaps[driver["DriverNumber"]] = gap if gap.total_seconds() >= 0 else driver["Q2"] - q2_pole
        elif pd.notna(driver["Q1"]):
            gap = driver["Q1"] - q3_pole
            pole_gaps[driver["DriverNumber"]] = gap if gap.total_seconds() >= 0 else driver["Q1"] - q1_pole
        else:
            # No qualifying time recorded — use full Q3 pole as a penalty
            pole_gaps[driver["DriverNumber"]] = q3_pole

    return pole_gaps


def get_teammate_qualifying_position(qualifying):
    """Return a dict mapping driver number → teammate's qualifying position."""
    teammate_pos = {}
    results = qualifying.results

    for _, driver in results.iterrows():
        for _, teammate in results.iterrows():
            if driver["DriverNumber"] == teammate["DriverNumber"]:
                continue
            if teammate["TeamName"] == driver["TeamName"]:
                teammate_pos[driver["DriverNumber"]] = teammate["Position"]
                break

    return teammate_pos


def get_qualifying_positions(qualifying):
    """Return a dict mapping driver number → qualifying classification position."""
    return {
        driver["DriverNumber"]: driver["Position"]
        for _, driver in qualifying.results.iterrows()
    }


def get_race_finishing_positions(race):
    """Return a dict mapping driver number → race finishing position."""
    return {
        driver["DriverNumber"]: driver["Position"]
        for _, driver in race.results.iterrows()
    }


def get_date_and_time(event):
    """Extract the UTC date and time of the Race session from an event row."""
    for i in range(1, 7):
        if event[f"Session{i}"] == "Race":
            dt = event[f"Session{i}DateUtc"]
            return dt.date(), dt.time()
    raise ValueError(f"No Race session found for event: {event['OfficialEventName']}")

---
## 5. Weather Fetching

Two separate functions handle historical vs. forecast weather:

- **`get_hourly_weather_info`** — calls the [Open-Meteo Historical Forecast API](https://open-meteo.com/en/docs/historical-forecast-api). Used for all completed races.
- **`predict_hourly_weather_info`** — calls the standard [Open-Meteo Forecast API](https://open-meteo.com/en/docs) with `past_days=1`. Used to generate pre-race prediction inputs before a race starts.

Both return the `hourly` block of the API response, which is passed directly into the weather helper functions above.

In [ ]:
def get_hourly_weather_info(event, date, time):
    """Fetch historical hourly weather for a completed race (±1 day window)."""
    info = track_locations[event["Location"].lower()]
    url = (
        f"https://historical-forecast-api.open-meteo.com/v1/forecast"
        f"?latitude={info['latitude']}&longitude={info['longitude']}"
        f"&start_date={date - timedelta(days=1)}&end_date={date + timedelta(days=1)}"
        f"&hourly=rain,relative_humidity_2m,apparent_temperature,wind_speed_10m"
    )
    return requests.get(url).json()["hourly"]


def predict_hourly_weather_info(event):
    """Fetch forecast hourly weather for an upcoming race (past_days=1 + 2-day forecast)."""
    info = track_locations[event["Location"].lower()]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={info['latitude']}&longitude={info['longitude']}"
        f"&hourly=rain,relative_humidity_2m,apparent_temperature,wind_speed_10m"
        f"&forecast_days=2&past_days=1"
    )
    return requests.get(url).json()["hourly"]

---
## 6. Main Collection Loop (2022–2025)

Iterates over the full calendar for each season, skipping:
- Pre-season testing events
- Rounds not yet run (future `EventDate`)
- 2022 rounds before the ground-effect era cutoff (rounds 1–18)

Each race produces a CSV at `{season}/{location}_race.csv` containing one row per driver. Location and driver IDs are assigned incrementally on first encounter and reused across all subsequent races.

In [ ]:
FIELDNAMES = [
    "DriverId", "TeamId", "Season", "RoundNumber", "RacesInGEEra",
    "LocationId", "RainBefore", "RainDuring", "WindSpeed", "ApparentTemp",
    "RelativeHumidity", "StartingPos", "QualifyingPos", "GapToPole",
    "TeammateQualifyingPos", "Retired", "FinishingPos",
]

In [ ]:
date_today = date.today()

driver_ids    = {}
driver_id_counter = 1

location_ids  = {}
location_id_counter = 1

races_in_ge_era = 1

for season in range(2022, 2026):
    for _, event in fastf1.get_event_schedule(season).iterrows():

        # Skip pre-season testing
        if event["EventFormat"] == "testing":
            continue

        # Skip rounds not yet run
        if event["EventDate"] > pd.Timestamp(date_today):
            continue

        # 2022: only include rounds from the ground-effect era cutoff (round 19+)
        if season == 2022 and event["RoundNumber"] < 19:
            continue

        # Normalise accented location names so they match track_locations keys
        location = event["Location"]
        location = location.replace("Montréal", "Montreal").replace("São Paulo", "Sao Paulo")
        event["Location"] = location

        # --- Session loading ---
        date_race, time_race = get_date_and_time(event)
        weather = get_hourly_weather_info(event, date_race, time_race)

        qualifying = fastf1.get_session(season, event["OfficialEventName"], "Q")
        qualifying.load(weather=True)
        race = fastf1.get_session(season, event["OfficialEventName"], "R")
        race.load(weather=True)

        # --- Feature extraction ---
        rain_before       = calculate_rain_amount_before(weather, time_race)
        rain_during       = calculate_rain_amount_during(weather, time_race)
        wind_speed        = calculate_wind_speed(weather, time_race)
        apparent_temp     = calculate_apparent_temp(weather, time_race)
        relative_humidity = calculate_relative_humidity(weather, time_race)

        finishing_pos       = get_race_finishing_positions(race)
        qualifying_pos      = get_qualifying_positions(qualifying)
        pole_gaps           = calculate_gap_to_pole_position(qualifying)
        teammate_qual_pos   = get_teammate_qualifying_position(qualifying)

        # --- Location ID assignment ---
        if location not in location_ids:
            location_ids[location] = location_id_counter
            location_id_counter += 1
        location_id = location_ids[location]

        # --- Write per-race CSV ---
        output_path = f"../{season}/{location}_race.csv"
        with open(output_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
            writer.writeheader()

            for _, driver in race.results.iterrows():
                # Assign driver ID on first encounter
                if driver["FullName"] not in driver_ids:
                    driver_ids[driver["FullName"]] = driver_id_counter
                    driver_id_counter += 1

                writer.writerow({
                    "DriverId":              driver_ids[driver["FullName"]],
                    "TeamId":                team_ids[driver["TeamName"]],
                    "Season":                season,
                    "RoundNumber":           event["RoundNumber"],
                    "RacesInGEEra":          races_in_ge_era,
                    "LocationId":            location_id,
                    "RainBefore":            rain_before,
                    "RainDuring":            rain_during,
                    "WindSpeed":             wind_speed,
                    "ApparentTemp":          apparent_temp,
                    "RelativeHumidity":      relative_humidity,
                    "StartingPos":           driver["GridPosition"],
                    "QualifyingPos":         qualifying_pos[driver["DriverNumber"]],
                    "GapToPole":             pd.to_timedelta(pole_gaps[driver["DriverNumber"]]).total_seconds(),
                    "TeammateQualifyingPos": teammate_qual_pos.get(driver["DriverNumber"], 0),
                    "Retired":               driver["Status"] not in ("Finished", "Lapped"),
                    "FinishingPos":          finishing_pos[driver["DriverNumber"]],
                })

        races_in_ge_era += 1
        print(f"✓ {season} R{event['RoundNumber']:02d} {location} → {output_path}")

---
## 7. Single-Race Backfill

Used to collect data for a specific completed race without re-running the full loop — for example when adding a late-season round or correcting a previously failed collection.

Update the `season`, `event_name`, `races_in_ge_era`, and `location_id` values before running.

In [ ]:
# --- Configuration ---
BACKFILL_SEASON       = 2025
BACKFILL_EVENT_NAME   = "Abu Dhabi"
BACKFILL_GE_ERA_COUNT = 91   # Cumulative race count for this round
BACKFILL_LOCATION_ID  = 22   # Pre-assigned location ID for Yas Island

# --- Load sessions ---
event = fastf1.get_event(BACKFILL_SEASON, BACKFILL_EVENT_NAME)

location = event["Location"]
location = location.replace("Montréal", "Montreal").replace("São Paulo", "Sao Paulo")
event["Location"] = location

qualifying = fastf1.get_session(BACKFILL_SEASON, event["OfficialEventName"], "Q")
qualifying.load()
race = fastf1.get_session(BACKFILL_SEASON, event["OfficialEventName"], "R")
race.load()

date_race, time_race = get_date_and_time(event)
weather = get_hourly_weather_info(event, date_race, time_race)

# --- Feature extraction ---
rain_before       = calculate_rain_amount_before(weather, time_race)
rain_during       = calculate_rain_amount_during(weather, time_race)
wind_speed        = calculate_wind_speed(weather, time_race)
apparent_temp     = calculate_apparent_temp(weather, time_race)
relative_humidity = calculate_relative_humidity(weather, time_race)

qualifying_pos    = get_qualifying_positions(qualifying)
pole_gaps         = calculate_gap_to_pole_position(qualifying)
teammate_qual_pos = get_teammate_qualifying_position(qualifying)
finishing_pos     = get_race_finishing_positions(race)

# --- Write CSV ---
output_path = f"../{BACKFILL_SEASON}/{location}_race.csv"
with open(output_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()

    for _, driver in race.results.iterrows():
        writer.writerow({
            "DriverId":              driver_ids[driver["FullName"]],
            "TeamId":                team_ids[driver["TeamName"]],
            "Season":                BACKFILL_SEASON,
            "RoundNumber":           event["RoundNumber"],
            "RacesInGEEra":          BACKFILL_GE_ERA_COUNT,
            "LocationId":            BACKFILL_LOCATION_ID,
            "RainBefore":            rain_before,
            "RainDuring":            rain_during,
            "WindSpeed":             wind_speed,
            "ApparentTemp":          apparent_temp,
            "RelativeHumidity":      relative_humidity,
            "StartingPos":           driver["GridPosition"],
            "QualifyingPos":         qualifying_pos[driver["DriverNumber"]],
            "GapToPole":             pd.to_timedelta(pole_gaps[driver["DriverNumber"]]).total_seconds(),
            "TeammateQualifyingPos": teammate_qual_pos.get(driver["DriverNumber"], 0),
            "Retired":               driver["Status"] not in ("Finished", "Lapped"),
            "FinishingPos":          finishing_pos[driver["DriverNumber"]],
        })

print(f"✓ Backfilled: {output_path}")

---
## 8. Pre-Race Prediction Input

Generates `race_prediction.csv` — the input file used by `evaluate.ipynb` to produce a predicted grid before a race starts. Run this after qualifying has concluded.

This cell uses `predict_hourly_weather_info` (forecast API) rather than the historical endpoint since the race hasn't happened yet. `FinishingPos` and `Retired` are omitted — they are the unknowns we're predicting.

> **Note:** `RacesInGEEra` and `LocationId` must be set manually to match the round being predicted.

In [ ]:
PREDICT_SEASON      = 2025
PREDICT_EVENT_NAME  = "Brazil"
PREDICT_GE_ERA_COUNT = 89
PREDICT_LOCATION_ID  = 21

PREDICT_FIELDNAMES = [
    "DriverId", "TeamId", "Season", "RoundNumber", "RacesInGEEra",
    "LocationId", "RainBefore", "RainDuring", "WindSpeed", "ApparentTemp",
    "RelativeHumidity", "StartingPos", "QualifyingPos", "GapToPole",
    "TeammateQualifyingPos",
]

# --- Load qualifying only (race hasn't started) ---
event = fastf1.get_event(PREDICT_SEASON, PREDICT_EVENT_NAME)

location = event["Location"]
location = location.replace("Montréal", "Montreal").replace("São Paulo", "Sao Paulo")
event["Location"] = location

qualifying = fastf1.get_session(PREDICT_SEASON, event["OfficialEventName"], "Q")
qualifying.load()

date_race, time_race = get_date_and_time(event)
weather = predict_hourly_weather_info(event)

# --- Feature extraction ---
rain_before       = calculate_rain_amount_before(weather, time_race)
rain_during       = calculate_rain_amount_during(weather, time_race)
wind_speed        = calculate_wind_speed(weather, time_race)
apparent_temp     = calculate_apparent_temp(weather, time_race)
relative_humidity = calculate_relative_humidity(weather, time_race)

qualifying_pos    = get_qualifying_positions(qualifying)
pole_gaps         = calculate_gap_to_pole_position(qualifying)
teammate_qual_pos = get_teammate_qualifying_position(qualifying)

# --- Write prediction input CSV ---
with open("../data/race_prediction.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=PREDICT_FIELDNAMES)
    writer.writeheader()

    for _, driver in qualifying.results.iterrows():
        writer.writerow({
            "DriverId":              driver_ids[driver["FullName"]],
            "TeamId":                team_ids[driver["TeamName"]],
            "Season":                PREDICT_SEASON,
            "RoundNumber":           event["RoundNumber"],
            "RacesInGEEra":          PREDICT_GE_ERA_COUNT,
            "LocationId":            PREDICT_LOCATION_ID,
            "RainBefore":            rain_before,
            "RainDuring":            rain_during,
            "WindSpeed":             wind_speed,
            "ApparentTemp":          apparent_temp,
            "RelativeHumidity":      relative_humidity,
            "StartingPos":           qualifying_pos[driver["DriverNumber"]],
            "QualifyingPos":         qualifying_pos[driver["DriverNumber"]],
            "GapToPole":             pd.to_timedelta(pole_gaps[driver["DriverNumber"]]).total_seconds(),
            "TeammateQualifyingPos": teammate_qual_pos.get(driver["DriverNumber"], 0),
        })

print(f"✓ Prediction input written → race_prediction.csv")